# 3. Non-Speech Recommendation

Uses the FAISS index from notebook 2.

1. Load `clap_embeddings.faiss` + `clap_metadata.json`
2. Keep only clips with `is_speech=False`
3. Pick 3 random non-speech queries
4. For each query, find the top-10 most similar **non-speech** clips

**Important:** in notebook 2, FAISS row `i` lines up with `metadata[i]`.  
The `id` field is the file UUID — not the FAISS row number.

## 0. Imports and paths

In [ ]:
import json
import random
from pathlib import Path

import faiss
import numpy as np
from IPython.display import Audio, display

ROOT = Path("..").resolve()
AUDIO_DIR = ROOT / "data" / "audio"
INDEX_PATH = ROOT / "data_index" / "clap_embeddings.faiss"
META_PATH = ROOT / "data_index" / "clap_metadata.json"

TOP_K = 10          # recommendations per query
SEARCH_K = 100      # fetch extra neighbors, then filter to non-speech
N_QUERIES = 3

print(f"index exists={INDEX_PATH.is_file()}")
print(f"meta  exists={META_PATH.is_file()}")

## 1. Load index + metadata

Same artifacts notebook 2 saved under `data_index/`.

In [ ]:
index = faiss.read_index(str(INDEX_PATH))
metadata = json.loads(META_PATH.read_text())

assert index.ntotal == len(metadata), "index size and metadata length must match"

# filename → full path (for Audio playback)
audio_path = {p.name: p for p in AUDIO_DIR.iterdir() if p.is_file()}

print(f"vectors: {index.ntotal}")
print(f"audio files on disk: {len(audio_path)}")

## 2. Non-speech pool

Remember each item’s **FAISS row** (`faiss_idx`) — that’s what we reconstruct and search with.

In [ ]:
non_speech = [
    {"faiss_idx": i, **meta}
    for i, meta in enumerate(metadata)
    if not meta["is_speech"]
]

print(f"non-speech clips: {len(non_speech)}")

## 3. Pick 3 random queries

In [ ]:
random.seed(42)
queries = random.sample(non_speech, N_QUERIES)

for q in queries:
    print(f"  faiss_idx={q['faiss_idx']:4d}  {q['filename']}")

## 4. Search: sound → similar non-speech sounds

1. Reconstruct each query vector from FAISS (`IndexFlatIP` supports this)
2. Search neighbors
3. Drop the query itself and anything tagged speech
4. Keep top-10

In [ ]:
def recommend(faiss_idx: int, k: int = TOP_K):
    """Top-k non-speech neighbors for the clip at this FAISS row."""
    query_vec = index.reconstruct(faiss_idx).astype("float32").reshape(1, -1)
    scores, ids = index.search(query_vec, SEARCH_K)

    hits = []
    for score, row in zip(scores[0], ids[0]):
        if row < 0 or row == faiss_idx:
            continue
        meta = metadata[int(row)]
        if meta["is_speech"]:
            continue
        hits.append({"score": float(score), **meta})
        if len(hits) == k:
            break
    return hits


results = []
for q in queries:
    recs = recommend(q["faiss_idx"])
    results.append({"query": q, "recommendations": recs})
    print(f"{q['filename']}: {len(recs)} recommendations")

## 5. Listen

Play each query and its top-10 matches.

In [ ]:
def play(filename: str):
    path = audio_path.get(filename)
    if path is None:
        print(f"  missing file: {filename}")
        return
    display(Audio(str(path)))


for block in results:
    q = block["query"]
    print(f"\n{'=' * 50}")
    print(f"Query: {q['filename']}")
    print("=" * 50)
    play(q["filename"])

    print("\nTop 10 non-speech recommendations:")
    for i, rec in enumerate(block["recommendations"], 1):
        print(f"\n{i}. {rec['filename']}  (score={rec['score']:.4f})")
        play(rec["filename"])